
# <span style="color:rgb(213,80,0)">Diffusion kurtosis imaging with DMRIMatlab</span>

As we discussed in the previous chapter, the diffusion tensor signal model does not apply when b\-values over $2,000s/mm^2$ are present. Beyond this point, the "kurtosis effect" plays an important role in the signal formation process that can "fool" the estimation of the diffusion tensor leading to physically inconsistent results. In the present chapter we will review how the kurtosis effect can be taken into account, how it can be quantitatively characterized, and what are its implications.


The kurtosis effect can be modeled in terms of a fourth order hyper\-symmetric tensor with 15 degrees of freedom, so that the diffusion signal in the logarithmic domain reads:


\[1\] $\log \left(S\left(b,\mathit{\mathbf{g}}\right)\right)=\log \left(S_0 \right)-b\sum_{i=1}^3 \sum_{j=1}^3 D_{\textrm{ij}} n_i n_j +\frac{b^2 }{6}{\textrm{MD}}^2 \sum_{i=1}^3 \sum_{j=1}^3 \sum_{k=1}^3 \sum_{l=1}^3 W_{\textrm{ijkl}} n_i n_j n_k n_l$


where $\mathit{\mathbf{g}}={\left\lbrack n_1 ,n_2 ,n_3 \right\rbrack }^T$ is the gradient direction applied to obtain the diffusion signal $S\left(b,\mathit{\mathbf{g}}\right)$ at the b\-value *b,* MD is the mean diffusivity introduced in the previous chapter, computed as 1/3 times the trace of $\left\lbrack D_{\textrm{ij}} \right\rbrack$ , and \[ $W_{\textrm{ijkl}}$ \] is the kurtosis tensor. Once again: due to the hyper\-symmetry of the kurtosis, only 15 up of the $3^4 =81$ components of $W$ are unique. The kurtosis tensor, together with the diffusion tensor, can represent the diffusion signal for larger b\-values.


 ***TO DO:*** Clear all variables and figures to start from zero (*Ctrl+ENTER*):


In [ ]:
addpath('..');
setup__DMRIMatlab_toolbox;
clear;
close('all');

## 1. Prepare the data

***BEFORE WE PROCEED***:

-  Note that we need, at the very least, the attenuation signal and the minimal information to intepret it (i.e. the gradients table and b\-values). Let's load a test volume (*Ctrl+ENTER*):

In [ ]:
whos -file test_data3.mat
% The minimum required information is: the attenuation signal atti, the
% gradients table gi and the b-values bi. Note their sizes mast match.
% Additionally, an optional mask whose field of view matches that of atti
% is useful to avoid unnecessary computations. Load them all:
load test_data3.mat


-  As opposed to the tensor model, the kurtosis model can use b\-values over $2,000s/mm^2$ , hence we will keep the entire data set.
-  However, it is usually assumed that the diffusion+kurtosis representation holds only up to $3,000s/mm^2$ , so larger b\-values might lead to uncertain results.
-  NOTE: <u>kurtosis imaging requires multi\-shell data</u>, hence, your data needs to provide at least 2 different b\-values with a grand total of at least 21 gradient directions (corresponding to 21 unknowns: 6 from the unique components of the DT, 15 for the unique components of the KT).

In [ ]:
unique(bi)',


***QUIZ:***


 *The* <samp>bi</samp> *vector in this case has a lot more unique b\-values. Is there a way to cluster them into actual "shells" i.e. groups of gradients with similar b\-values?*

<pre>
help auto_detect_shells
</pre>

In [ ]:
% There are values not suitable for DTI
size(gi),
size(bi),
size(atti),

## 2. Estimate the diffusion tensor and the kurtosis tensor

Now we are ready to estimate the diffusion tensor and the kurtosis tensor, both of them together, with the proper function. The expected input is an attenuation volume, and the expected output is a DKT volume (and a DTI volume). Hence, the function is named <samp>atti2dkt</samp> (*Ctrl+ENTER*):


In [ ]:
% atti, gi and bi are the minimum required information, and therefore they
% are mandatory inputs that are passed as "positional arguments". atti2dkt
% has two additional mandatory inputs that we will left empty by now. All
% other input arguments are optional and are passed as 'key', value pairs:
tic;
[dkt,dti,S0] = atti2dkt( atti, gi, bi, [], [], ...
    'mask', mask, 'nconst', 60 );
fprintf(1,'It took %1.3f seconds to compute the kurtosis tensor\n',toc);


***QUIZ:***


*What do input arguments 4 and 5 stand for? How could you obtain them if you wanted to?*

<pre>
help atti2dkt
</pre>

Let's take a look to the sizes of the outputs returned:


In [ ]:
size(dkt),
size(dti),
size(S0)


As promised, each voxel of the <samp>dkt</samp> volume contains a 15\-component vector with the 15 unique components of the kurtosis tensor. The second output, <samp>dti</samp>, has exactly the same meaning as if it had been estimated with <samp>atti2dti</samp> as in the previous chapter (<u>but it was estimated in a completely different way!</u>). Let's check:


In [ ]:
[u1,~,~,l1,l2,l3] = dti2spectrum( dti, ...
    'mask', mask );
colored = spectrum2colorcode(u1,l1,l2,l3,'mask',mask);
close(figure(1));
hf1 = figure(1);
set(hf1,'Position',[1,1,1200,400]);
slices = [4,6,7,8,9,11];
plotSlicedVolume_demo1( hf1, colored, mask, slices, true );
title('Color-coding');


***QUIZ:***


*What are the mathematical techniques used to estimate the diffusion and the kurtosis tensor? What are the optional parameters that control these techniques? Try to play with 'qp', 'wls' and 'nconst' and check the results.*

<pre>
help atti2dkt
</pre>

## 3. Reconstruct the signal from the model

Eq. \[1\] provides a model to compute the diffusion signal from the diffusion tensor and the kurtosis tensor, so it should be possible to go forward and compute a reconstruction of the attenuation signal, $\frac{S\left(b,\mathit{\mathbf{g}}\right)}{S_0 }$ , once the two tensors are known. This is exactly what <samp>dkt2atti</samp> does:

<pre>
help dkt2atti
</pre>

We just need to tell the function the b\-values and gradient directions for which we want to reconstruct the signal. Let's use those in the original gradients table (*Ctrl+ENTER*):


In [ ]:
atti_reconst = dkt2atti( dkt, dti, S0, gi, bi, 'mask', mask );


***QUIZ:***


 *What is the S0 input used for? What is its exact meaning? Try to change the value of 'estimS0' in* <samp>atti2dkt</samp> *and repeat all the experiments.*

<pre>
help atti2dkt
help dkt2atti
</pre>

To check the actual impact of the kurtosis tensor in the reconstruction, we will use a simple trick: just pass a null kurtosis tensor so that the reconstruction is merely based on the diffusion tensor:


In [ ]:
atti_reconst2 = dkt2atti( zeros(size(dkt)), dti, S0, gi, bi, 'mask', mask );


***QUIZ:***


*Is there any other way to reconstruct a signal at desired directions/b\-values from a diffusion tensor?*

<pre>
help dti2signal
</pre>

No we will compare several DWI channels (gradient/b\-value combinations) of the original attenuation signal, the one reconstructed just from the diffusion tensor and the one reconstructed from the whole diffusion+kurtosis model:


In [ ]:
ch = [34,67,23,134,81,167];
close(figure(2));
hf2 = figure(2);
set(hf2,'Position',[1,1,2000,900]);
slice = 9;

for c=1:length(ch) % For each DWI channel:
    % ----
    IMG1 = ( atti(:,:,slice,ch(c)).*mask(:,:,slice) )';           % Original signal
    IMG2 = ( atti_reconst2(:,:,slice,ch(c)).*mask(:,:,slice) )';  % Reconstructed from DTI
    IMG3 = ( atti_reconst(:,:,slice,ch(c)).*mask(:,:,slice) )';   % Reconstructed from DTI+DKT
    % ----
    subplot(1,length(ch),c);
    imshow([IMG3;IMG2;IMG1]);
    axis('xy');
    title(   sprintf( 'b=%1.0f', bi(ch(c)) ), 'FontSize', 24   );
    % ----
end
drawnow;



Some things to note:

-  Note the "denoising" effect of the reconstruction process with respect to the original signal. This makes sense, since the whole set of channels is reconstructed from as few as 21 degrees of freedom.
-  The impact of the kurtosis tensor now becomes clear: without it, the reconstructed signal rapidly vanishes to 0 so that for large b\-values it has much smaller values than the original one. The kurtosis term, being a positive addend to the argument of the exponential in eq. \[1\], corrects this behavior and allows a slower decay of the reconstructed signal.
-  Yet, even when the decay of the attenuation signal is slower with the kurtosis term, the attenuation signal should always be monotonically decreasing to be physically consistent.

***QUIZ:***


*Is the latter physical constraint always enforced by the algorithm?*

<pre>
help atti2dkt
</pre>

## **4. Quantify kurtosis**

In the light of the previous section, it seem clear that the kurtosis effect plays an important role in the description of the attenuation signal, but, how can we quantify this impact at each voxel? The kurtosis at a given spatial orientation (i.e. gradient direction) $\mathit{\mathbf{n}}={\left\lbrack n_1 ,n_2 ,n_3 \right\rbrack }^T$ is defined as:


\[2\] $K\left(\mathit{\mathbf{n}}\right)=\frac{{\textrm{MD}}^2 }{{\left(\sum_{i=1}^3 \sum_{j=1}^3 D_{\textrm{ij}} n_i n_j \right)}^2 }$ $\sum_{i=1}^3 \sum_{j=1}^3 \sum_{k=1}^3 \sum_{l=1}^3 W_{\textrm{ijkl}} n_i n_j n_k n_l$


From here, we can define:

-  The mean kurtosis MK as the averaged kurtosis for all spatial orientations.
-  The axial kurtosis AK as the kurtosis at the main diffusion direction of the diffusion tensor.
-  The radial kurtosis RK as the averaged kurtosis for all directions perpendicular to the main direction of the diffusion tensor.

In practice, if $\left\lbrace {\mathit{\mathbf{u}}}_1 ,{\mathit{\mathbf{u}}}_2 ,{\mathit{\mathbf{u}}}_3 \right\rbrace$ are the three (ordered, see previous chapter) eigenvectors of the diffusion tensor, we compute:

-  $\displaystyle \textrm{MK}=\frac{\left(K\left({\mathit{\mathbf{u}}}_1 \right)+K\left({\mathit{\mathbf{u}}}_2 \right)+K\left({\mathit{\mathbf{u}}}_3 \right)\right)}{3}$
-  AK = $K\left({\mathit{\mathbf{u}}}_1 \right)$
-  RK = $\frac{\left(K\left({\mathit{\mathbf{u}}}_2 \right)+K\left({\mathit{\mathbf{u}}}_3 \right)\right)}{2}$

which are all computed at a time with  <samp>dkt2kurtosis</samp> (*Ctrl+ENTER*):


In [ ]:
[MK,AK,RK] = dkt2kurtosis( dkt, dti, 'mask', mask );
close(figure(3));
hf3 = figure(3);
set(hf3,'Position',[1,1,2000,400]);
plotSlicedVolume_demo1(hf3,MK,mask,slices,true,[0,1.5]);
colormap(parula);
colorbar;
title('MK','FontSize',24);
close(figure(4));
hf4 = figure(4);
set(hf4,'Position',[1,1,2000,400]);
plotSlicedVolume_demo1(hf4,AK,mask,slices,true,[0,1.25]);
colormap(parula);
colorbar;
title('AK','FontSize',24);
close(figure(5));
hf5 = figure(5);
set(hf5,'Position',[1,1,2000,400]);
plotSlicedVolume_demo1(hf5,RK,mask,slices,true,[0,2.0]);
colormap(parula);
colorbar;
title('RK','FontSize',24);


***QUIZ:***

-   *Try to change the parameters of the estimation algorithm,* <samp>atti2dkt</samp>*, and check the results in all the outputs of this demo.*

[Go back to index page](./tour_00index.ipynb)


[Previous](./tour_03dtmri.ipynb)


[Next](./tour_04fwvf.ipynb)
